# Architecture hotspots

Find busy containers, deep ownership paths, and dominant semantic types.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

Loaded 42 SysML files from /Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model


In [2]:
from IPython.display import display

semantic_elements = (
    list(model.elements(syside.Usage, include_subtypes=True))
    + list(model.elements(syside.Definition, include_subtypes=True))
)

def safe_name(element):
    value = element.qualified_name or element.name or element.declared_name
    return str(value) if value else '<unnamed>'

def ownership_depth(element):
    depth, current, seen = 0, element.owner, set()
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        depth += 1
        current = getattr(current, 'owner', None)
    return depth

containers = []
for element in semantic_elements:
    children = [child for child in element.owned_elements if isinstance(child, (syside.Usage, syside.Definition))]
    if children:
        containers.append((len(children), type(element).__name__, safe_name(element)))

print('Top containers by directly owned semantic elements')
display(sorted(containers, reverse=True)[:25])

print('Deepest ownership paths')
display(sorted(
    ((ownership_depth(item), type(item).__name__, safe_name(item)) for item in semantic_elements),
    reverse=True,
)[:25])

print('Most common semantic types')
Counter(type(item).__name__ for item in semantic_elements).most_common(30)

Top containers by directly owned semantic elements


[(19,
  'RequirementUsage',
  'memo_examples_gpca_pump_model_catalog_gpca_requirements::reqWatchdogSafeState'),
 (19,
  'RequirementUsage',
  'memo_examples_gpca_pump_model_catalog_gpca_requirements::reqDoorOpen'),
 (18,
  'RequirementUsage',
  'memo_examples_gpca_pump_model_catalog_gpca_requirements::reqPowerLossRetention'),
 (18,
  'RequirementUsage',
  'memo_examples_gpca_pump_model_catalog_gpca_requirements::reqPost'),
 (18,
  'RequirementUsage',
  'memo_examples_gpca_pump_model_catalog_gpca_requirements::reqDrugLibraryLimits'),
 (18,
  'ActionUsage',
  'memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow'),
 (17,
  'ViewUsage',
  'memo_examples_gpca_pump_model_views_logical_system_decomposition_view::gpcaSystemDecompositionView'),
 (16,
  'ViewUsage',
  'memo_examples_gpca_pump_model_views_software_software_architecture_view::gpcaSoftwareArchitectureView'),
 (16,
  'ViewUsage',
  'memo_examples_gpca_pump_model_views_risk_risk_chain_view::gpcaRiskChain

Deepest ownership paths


[(5, 'ReferenceUsage', 'sensorStatus'),
 (5, 'ReferenceUsage', 'sensorStatus'),
 (5, 'ReferenceUsage', 'flowCommand'),
 (5, 'ReferenceUsage', 'flowCommand'),
 (5, 'ReferenceUsage', 'authorizedBolus'),
 (5, 'ReferenceUsage', 'authorizedBolus'),
 (5, 'ReferenceUsage', 'alarmSignal'),
 (5, 'ReferenceUsage', 'alarmSignal'),
 (5, 'ReferenceUsage', 'actuation'),
 (5, 'ReferenceUsage', 'actuation'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>'),
 (4, 'AttributeUsage', '<unnamed>')]

Most common semantic types


[('AttributeUsage', 5056),
 ('ConnectionUsage', 691),
 ('PartUsage', 425),
 ('ReferenceUsage', 174),
 ('RequirementUsage', 105),
 ('ItemUsage', 49),
 ('ActionUsage', 44),
 ('SuccessionAsUsage', 29),
 ('VerificationCaseUsage', 27),
 ('ViewUsage', 26),
 ('InterfaceUsage', 14),
 ('StateUsage', 10),
 ('AllocationUsage', 6),
 ('ActionDefinition', 6),
 ('FlowUsage', 5),
 ('ItemDefinition', 5),
 ('UseCaseUsage', 1),
 ('PortDefinition', 1),
 ('ConjugatedPortDefinition', 1),
 ('InterfaceDefinition', 1)]